# Reproducing Sleep Validation Methodology

This notebook reproduces the statistical methodology used in the research paper:
**"Validation of an automated wireless system to monitor sleep in healthy adults"** (Shambroom et al., 2012).

We will use a public dataset (Sleep-EDF from PhysioNet) to act as our primary gold-standard Polysomnography (PSG1). To reproduce the inter-rater and device comparisons, we will simulate a second human scorer (PSG2) and an automated wireless system (WS) by introducing characteristic errors observed in the paper (e.g., WS sometimes confusing Wake with Light sleep).

## 1. Setup and Installation
First, we install and import the necessary libraries. We use `mne` to fetch the Sleep-EDF dataset, `scikit-learn` for agreement metrics, and `pingouin` for advanced statistical tests like Repeated Measures ANOVA and Intraclass Correlation (ICC).

In [ ]:
!pip install mne yasa scikit-learn pingouin pandas matplotlib seaborn -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
import pingouin as pg

# Ensure reproducible results
np.random.seed(42)

## 2. Data Loading (Public Dataset)
We load a sample from the **Sleep-EDF** dataset. The sleep stages are typically scored as:
0: Wake, 1: N1 (Light), 2: N2 (Light), 3: N3 (Deep), 4: N4 (Deep), 5: REM.

For this methodology, following the paper, we group N1 and N2 into **Light Sleep**, and N3 and N4 into **Deep Sleep**.

In [ ]:
# Fetch a single subject's data from Sleep-EDF
data_path = mne.datasets.sleep_physionet.age.fetch_data(subjects=[0], recording=[1])[0]
annot = mne.read_annotations(data_path[1])

# Map Sleep-EDF annotations to our simplified stages: Wake (W), Light (L), Deep (D), REM (R)
annotation_desc_2_event_id = {
    'Sleep stage W': 0,    # Wake
    'Sleep stage 1': 1,    # Light (Stage 1)
    'Sleep stage 2': 1,    # Light (Stage 2)
    'Sleep stage 3': 2,    # Deep (Stage 3)
    'Sleep stage 4': 2,    # Deep (Stage 4)
    'Sleep stage R': 3,    # REM
    'Movement time': 0,    # Treat movement as Wake
    'Sleep stage ?': -1    # Unknown
}

# Create an array of epoch-by-epoch scores (30-second epochs)
# The annotations have duration, we convert them to a sequence of 30s epochs
epochs_psg1 = []
for desc, dur in zip(annot.description, annot.duration):
    if desc in annotation_desc_2_event_id:
        val = annotation_desc_2_event_id[desc]
        # Calculate how many 30s epochs this annotation covers
        n_epochs = int(dur // 30)
        epochs_psg1.extend([val] * n_epochs)

epochs_psg1 = np.array(epochs_psg1)
epochs_psg1 = epochs_psg1[epochs_psg1 != -1] # Remove unknown epochs

stage_names = ['Wake', 'Light', 'Deep', 'REM']
print(f"Loaded {len(epochs_psg1)} epochs of PSG1 data.")

## 3. Simulating Scorers and Devices
Since we only have one ground truth (PSG1), we will simulate a second human scorer (PSG2) and the Wireless System (WS) based on the agreement rates found in the paper:
- **PSG1 vs PSG2**: High agreement (~83%).
- **PSG1 vs WS**: Moderate agreement (~75%), with WS occasionally scoring Wake as Light sleep.

In [ ]:
def simulate_scorer(base_scores, agreement_rate, bias_to_light=False):
    """Simulates a new scorer by introducing random errors to the base scores."""
    new_scores = base_scores.copy()
    n_epochs = len(new_scores)
    
    # Determine which epochs will be altered based on agreement rate
    n_errors = int(n_epochs * (1 - agreement_rate))
    error_indices = np.random.choice(n_epochs, n_errors, replace=False)
    
    for idx in error_indices:
        current_stage = new_scores[idx]
        possible_stages = [0, 1, 2, 3]
        possible_stages.remove(current_stage)
        
        if bias_to_light and current_stage == 0 and np.random.rand() < 0.6:
            # WS system tendency to score Wake as Light sleep
            new_scores[idx] = 1
        else:
            # Random misclassification
            new_scores[idx] = np.random.choice(possible_stages)
            
    return new_scores

# Create multiple subject nights to allow for summary statistics testing (e.g. 10 simulated subjects)
# For the epoch-by-epoch analysis we will just pool all data, but for summary stats we need subjects.
subjects_data = []
for subj_id in range(1, 11):
    # We slightly shift the base PSG1 to simulate different subjects
    subj_psg1 = np.roll(epochs_psg1, shift=np.random.randint(-100, 100))
    subj_psg2 = simulate_scorer(subj_psg1, agreement_rate=0.83)
    subj_ws = simulate_scorer(subj_psg1, agreement_rate=0.75, bias_to_light=True)
    
    subjects_data.append({
        'Subject': subj_id,
        'PSG1': subj_psg1,
        'PSG2': subj_psg2,
        'WS': subj_ws
    })

# Pooled epochs for Section 4
pooled_psg1 = np.concatenate([d['PSG1'] for d in subjects_data])
pooled_psg2 = np.concatenate([d['PSG2'] for d in subjects_data])
pooled_ws = np.concatenate([d['WS'] for d in subjects_data])

print(f"Simulated 10 subjects. Total pooled epochs: {len(pooled_psg1)}")

## 4. Epoch-by-Epoch Analysis
We calculate the **Percentage Agreement** and **Cohen's Kappa** for sleep stages between the systems. Cohen's Kappa measures the agreement beyond what would be expected by chance.

In [ ]:
def calculate_agreement(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred) * 100
    kappa = cohen_kappa_score(y_true, y_pred)
    return acc, kappa

acc_psg_ws, kappa_psg_ws = calculate_agreement(pooled_psg1, pooled_ws)
acc_psg_psg2, kappa_psg_psg2 = calculate_agreement(pooled_psg1, pooled_psg2)

print("Epoch-by-epoch sleep stage agreement:\n")
print(f"WS vs PSG1:   Agreement = {acc_psg_ws:.1f}%, Cohen's Kappa = {kappa_psg_ws:.2f}")
print(f"PSG1 vs PSG2: Agreement = {acc_psg_psg2:.1f}%, Cohen's Kappa = {kappa_psg_psg2:.2f}")

### Contingency Tables
Figure 3 of the paper shows contingency tables (confusion matrices) identifying specific areas of disagreement. We visualize these using heatmaps.

In [ ]:
def plot_contingency_table(y_true, y_pred, title, ax):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=stage_names, yticklabels=stage_names, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_xlabel('System B (Predicted)')
    ax.set_ylabel('System A (Reference)')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_contingency_table(pooled_psg1, pooled_ws, "PSG1 vs WS", axes[0])
plot_contingency_table(pooled_psg1, pooled_psg2, "PSG1 vs PSG2", axes[1])
plt.tight_layout()
plt.show()

## 5. Summary Sleep Measures
We calculate nightly sleep statistics like Total Sleep Time (TST), Sleep Efficiency (SE), and Wake After Sleep Onset (WASO). Note: 1 epoch = 30 seconds (0.5 minutes).

In [ ]:
def calculate_summary_measures(epochs):
    total_epochs = len(epochs)
    sleep_epochs = epochs[epochs > 0]
    tst = len(sleep_epochs) * 0.5 # Total sleep time in minutes
    se = (len(sleep_epochs) / total_epochs) * 100 # Sleep efficiency (%)
    
    # Wake after sleep onset (WASO)
    sleep_indices = np.where(epochs > 0)[0]
    waso = 0
    if len(sleep_indices) > 0:
        first_sleep = sleep_indices[0]
        last_sleep = sleep_indices[-1]
        waso_epochs = np.sum(epochs[first_sleep:last_sleep] == 0)
        waso = waso_epochs * 0.5
        
    return tst, se, waso

summary_stats = []
for d in subjects_data:
    for system in ['PSG1', 'PSG2', 'WS']:
        tst, se, waso = calculate_summary_measures(d[system])
        summary_stats.append({
            'Subject': d['Subject'],
            'System': system,
            'TST': tst,
            'SE': se,
            'WASO': waso
        })

df_summary = pd.DataFrame(summary_stats)
df_summary.head(9)

## 6. Statistical Analysis (ANOVA and ICC)
To determine if there are significant differences between the systems, we use a **Repeated Measures ANOVA**. We also calculate the **Intraclass Correlation Coefficient (ICC)** to evaluate the agreement on summary measures.

In [ ]:
# Repeated Measures ANOVA for Total Sleep Time (TST)
anova_res = pg.rm_anova(dv='TST', within='System', subject='Subject', data=df_summary, detailed=True)
print("Repeated Measures ANOVA for TST:")
print(anova_res[['Source', 'ddof1', 'ddof2', 'F', 'p-unc']])
print("\n(If p < 0.05, there is a significant difference in TST across the scoring systems.)")

# Intraclass Correlation Coefficient (ICC) for TST
# ICC measures the reliability of ratings for the same subjects across different systems
icc_res = pg.intraclass_corr(data=df_summary, targets='Subject', raters='System', ratings='TST')
icc_val = icc_res.loc[icc_res['Type'] == 'ICC3', 'ICC'].values[0]
print(f"\nIntraclass Correlation (ICC) for TST across systems: {icc_val:.3f}")
print("(Values > 0.90 indicate excellent reliability/agreement.)")

### Visualization: Scatter Plots
Finally, we can visualize the per-night agreement between WS and PSG1 for TST (similar to Figure 5 in the paper).

In [ ]:
df_pivot = df_summary.pivot(index='Subject', columns='System', values='TST')

plt.figure(figsize=(6, 6))
sns.scatterplot(x=df_pivot['PSG1'], y=df_pivot['WS'])
plt.plot([min(df_pivot['PSG1']), max(df_pivot['PSG1'])], 
         [min(df_pivot['PSG1']), max(df_pivot['PSG1'])], 
         color='red', linestyle='--') # Line of perfect agreement

plt.title('Total Sleep Time (min): WS vs PSG1')
plt.xlabel('PSG1 (min)')
plt.ylabel('Wireless System (min)')
plt.grid(True)
plt.show()